In [13]:
import pandas as pd
import numpy as np

patients = pd.read_csv('content/patients.csv')
treatments = pd.read_csv('content/treatments.csv')
adverse_reactions = pd.read_csv('content/adverse_reactions.csv')
treatments_cut = pd.read_csv('content/treatments_cut.csv')

### Data Quality Dimensions

- Completeness -> is data missing?
- Validity -> is data invalid -> negative height -> duplicate patient id
- Accuracy -> data is valid but not accurate -> weight -> 1kg
- Consistency -> both valid and accurate but written differently -> New Youk and NY

### Order of severity

Completeness <- Validity <- Accuracy <- Consistency

### Data Cleaning Order

1. Quality -> Completeness
2. Tidiness
3. Quality -> Validity
4. Quality -> Accuracy
5. Quality -> Consistency

#### Steps involved in Data cleaning
- Define
- Code
- Test

`Always make sure to create a copy of your pandas dataframe before you start the cleaning process`

In [14]:
patients_df = patients.copy()
treatments_df = treatments.copy()
treatments_cut_df = treatments_cut.copy()
adverse_reactions_df = adverse_reactions.copy()

### Define

- replace all missing values of patients df with no data
- sub hba1c_start from hba1c_end to get all the change values
- in patients table we will use regex to separate email and phone

In [15]:
# code
patients_df.fillna('No data',inplace=True)

C:\Users\Aayu0\AppData\Local\Temp\ipykernel_23872\115453354.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No data' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  patients_df.fillna('No data',inplace=True)


In [16]:
# test
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    object 
 2   given_name    503 non-null    object 
 3   surname       503 non-null    object 
 4   address       503 non-null    object 
 5   city          503 non-null    object 
 6   state         503 non-null    object 
 7   zip_code      503 non-null    object 
 8   country       503 non-null    object 
 9   contact       503 non-null    object 
 10  birthdate     503 non-null    object 
 11  weight        503 non-null    float64
 12  height        503 non-null    int64  
 13  bmi           503 non-null    float64
dtypes: float64(2), int64(2), object(10)
memory usage: 55.1+ KB


In [17]:
treatments.head()

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,veronika,jindrová,41u - 48u,-,7.63,7.20,NaN
1,elliot,richardson,-,40u - 45u,7.56,7.09,0.97
2,yukitaka,takenaka,-,39u - 36u,7.68,7.25,NaN
3,skye,gormanston,33u - 36u,-,7.97,7.62,0.35
4,alissa,montez,-,33u - 29u,7.78,7.46,0.32


In [18]:
# --------------------------------------------------
# Calculate HbA1c Change (Before vs After Treatment)
# --------------------------------------------------

# HbA1c change for the full treatments dataset
# Positive value  -> HbA1c reduced (improvement)
# Negative value  -> HbA1c increased (worsening)
treatments_df['hba1c_change'] = (
    treatments_df['hba1c_start'] - treatments_df['hba1c_end']
)

# HbA1c change for the filtered / cut treatments dataset
# Same logic applied to a subset of data
treatments_cut_df['hba1c_change'] = (
    treatments_cut_df['hba1c_start'] - treatments_cut_df['hba1c_end']
)


In [19]:
# test
treatments_cut_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    70 non-null     object 
 1   surname       70 non-null     object 
 2   auralin       70 non-null     object 
 3   novodra       70 non-null     object 
 4   hba1c_start   70 non-null     float64
 5   hba1c_end     70 non-null     float64
 6   hba1c_change  70 non-null     float64
dtypes: float64(3), object(4)
memory usage: 4.0+ KB


In [20]:
patients.head()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
0,1,female,Zoe,Wellish,576 Brown Bear Drive,Rancho California,California,92390.0,United States,951-719-9170ZoeWellish@superrito.com,7/10/1976,121.7,66,19.6
1,2,female,Pamela,Hill,2370 University Hill Road,Armstrong,Illinois,61812.0,United States,PamelaSHill@cuvox.de+1 (217) 569-3204,4/3/1967,118.8,66,19.2
2,3,male,Jae,Debord,1493 Poling Farm Road,York,Nebraska,68467.0,United States,402-363-6804JaeMDebord@gustr.com,2/19/1980,177.8,71,24.8
3,4,male,Liêm,Phan,2335 Webster Street,Woodbridge,NJ,7095.0,United States,PhanBaLiem@jourrapide.com+1 (732) 636-8246,7/26/1951,220.9,70,31.7
4,5,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303.0,United States,334-515-7487TimNeudorf@cuvox.de,2/18/1928,192.3,27,26.1


In [21]:
import re

In [22]:
def find_contact_details(text: str) -> tuple:
    
    # ! Step 1: Handle Missing Values (NaN check)
    # Agar value NaN hai, to directly return kar do
    if pd.isna(text):
        return np.nan
    

    # ! Step 2: Create Regex Pattern for Phone Number
    # Ye pattern phone number detect karega (with/without country code)
    phone_number_pattern = re.compile(
        r"(\+[\d]{1,3}\s)?(\(?[\d]{3}\)?\s?-?[\d]{3}\s?-?[\d]{4})"
    )
    

    # ! Step 3: Extract Phone Number using regex
    # findall → list of matches return karega
    phone_number = re.findall(phone_number_pattern, text)


    # ! Step 4: Handle Different Cases of Matches
    
    # Case 1: Agar koi phone number mila hi nahi
    if len(phone_number) <= 0:
        phone_number = np.nan

    # Case 2: Agar multiple parts mile (country code + number)
    # Example: +91 9876543210 → tuple me aayega
    elif len(phone_number) >= 2:
        phone_number = phone_number[1]

    # Case 3: Normal case → ek hi number mila
    else:
        phone_number = phone_number[0]


    # ! Step 5: Extract Email
    # Phone number ko text se remove kar dete hain
    # Jo bacha hai → wo mostly email hoga
    possible_email_add = re.sub(phone_number_pattern, "", text).strip()


    # ! Step 6: Return Output
    # Tuple me (phone_number, email) return kar rahe hain
    return phone_number, possible_email_add

In [23]:
# ! Extract phone number from contact column
patients_df['phone'] = (
    patients_df['contact']
    .apply(lambda x: find_contact_details(x))   # returns (phone, email)
    .apply(lambda x: x[0])                      # extract phone
)

# ! Extract email from contact column
patients_df['email'] = (
    patients_df['contact']
    .apply(lambda x: find_contact_details(x))   # returns (phone, email)
    .apply(lambda x: x[1])                      # extract email
)

In [24]:
patients_df.drop(columns='contact',inplace=True)

In [28]:
rows = []

for item in patients['contact']:

    # ! Handle NaN values
    if pd.isna(item):
        rows.append({'phone': None, 'email': None})
        continue

    phone_match = re.search(r'(\d{3}[-\.\s]??\d{3}[-\.\s]??\d{4}|\(\d{3}\)\s*\d{3}[-\.\s]??\d{4}|\d{3}[-\.\s]??\d{4})', item)
    email_match = re.search(r'([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+)', item)

    phone = phone_match.group(0) if phone_match else None
    email = email_match.group(1) if email_match else None

    rows.append({'phone': phone, 'email': email})

df = pd.DataFrame(rows)

In [29]:
treatments_df = pd.concat([treatments_df,treatments_cut_df])

In [30]:
treatments_df = treatments_df.melt(
    id_vars=['given_name', 'surname', 'hba1c_start', 'hba1c_end', 'hba1c_change'],
    var_name='type',
    value_name='dosage_range'
)

In [ ]:
# ! Extract starting dosage (before '-')
treatments_df['dosage_start'] = (
    treatments_df['dosage_range']
    .str.split('-')          # split "10-20u" → ["10", "20u"]
    .str.get(0)             # get first value → "10"
)

# ! Extract ending dosage (after '-')
treatments_df['dosage_end'] = (
    treatments_df['dosage_range']
    .str.split('-')          # split "10-20u"
    .str.get(1)             # get second value → "20u"
)

In [ ]:
treatments_df.drop(columns='dosage_range',inplace=True)

In [ ]:
# ! Remove 'u' from starting dosage (e.g., "10u" → "10")
treatments_df['dosage_start'] = (
    treatments_df['dosage_start']
    .str.replace('u', '')      # remove unit character
)

# ! Remove 'u' from ending dosage (e.g., "20u" → "20")
treatments_df['dosage_end'] = (
    treatments_df['dosage_end']
    .str.replace('u', '')      # clean text
)

In [ ]:
# ! Convert starting dosage to integer (string → int)
treatments_df['dosage_start'] = (
    treatments_df['dosage_start']
    .astype('int')           # "10" → 10
)

# ! Convert ending dosage to integer (string → int)
treatments_df['dosage_end'] = (
    treatments_df['dosage_end']
    .astype('int')           # "20" → 20
)

In [ ]:
treatments_df

In [ ]:
# ! Convert starting dosage to integer (string → int)
treatments_df['dosage_start'] = (
    treatments_df['dosage_start']
    .astype('int')           # "10" → 10
)

# ! Convert ending dosage to integer (string → int)
treatments_df['dosage_end'] = (
    treatments_df['dosage_end']
    .astype('int')           # "20" → 20
)
